# Bloom-Text-0.1 — Continue pretraining

Load the current published experimental checkpoint and its **original tokenizer** from a single Hugging Face revision. Train on a larger TinyStories subset plus the existing public-domain corpus, evaluate the parent and candidate on identical validation windows, and save candidates to Google Drive.

1. Choose an authorized T4 GPU runtime in Colab.
2. Run the cells in order. Approve Drive access only if you want persistent checkpoints.
3. Compare validation loss and generated samples. Review the saved candidate before publishing.

The parent does not include optimizer state; this is a new optimizer initialized from existing weights, not exact resumption. No automatic publication or paid compute provisioning. The current live checkpoint is left untouched.

In [ ]:
# 1. Clone Centauri Bloom (main) and install dependencies
import os, sys, subprocess
from pathlib import Path

if Path('pyproject.toml').is_file() and Path('bloom/model.py').is_file():
    REPO = Path.cwd()
elif Path('/content/centauri-bloom/pyproject.toml').is_file():
    REPO = Path('/content/centauri-bloom')
else:
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/InterstellarNerd8760/centauri-bloom.git', '/content/centauri-bloom'], check=True)
    REPO = Path('/content/centauri-bloom')

os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tokenizers', 'datasets'], check=True)
subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], check=True)

In [ ]:
# 2. Configure run & mount Google Drive for periodic SafeTensors checkpoints
import json, torch
from pathlib import Path

from datetime import datetime, timezone
RUN_NAME = 'bloom-continue-' + datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
TINY_SMOKE_TEST = False        # Set True to test locally on CPU in a few seconds
UPLOAD_TO_HF = False            # Upload final checkpoint to HF if Colab secret HF_TOKEN is set
HF_REPO_ID = 'Centauri-Bloom/Bloom-Text-0.1'
USE_TINYSTORIES = True         # Mix roneneldan/TinyStories (cdla-sharing-1.0) with public-domain-v1
TINYSTORIES_MAX_STORIES = 100_000
TINYSTORIES_VAL_STORIES = 500

STEPS = 1000                  # Additional optimizer steps; adjust after evaluation
BATCH_SIZE = 16
GRAD_ACCUM = 2
LEARNING_RATE = 1e-4
WARMUP_STEPS = 50
SAVE_EVERY = 250
EVAL_BATCHES = 32
SEED = 43

if TINY_SMOKE_TEST:
    raise ValueError('Use the original notebook for a fresh CPU smoke test.')

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and not TINY_SMOKE_TEST:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = Path('/content/drive/MyDrive/centauri-bloom-runs') / RUN_NAME
else:
    REPO = globals().get('REPO', Path.cwd())
    OUT_DIR = REPO / 'runs' / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=False)

DEVICE = 'cuda' if torch.cuda.is_available() and not TINY_SMOKE_TEST else 'cpu'
if DEVICE != 'cuda':
    raise RuntimeError('Select an authorized GPU runtime before training.')
print('Device:', DEVICE, '| Output dir:', OUT_DIR)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 3. Prepare corpus and restore the original tokenizer from the pinned parent
import hashlib, subprocess
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from tokenizers.trainers import BpeTrainer

PD_DIR = REPO / 'runs' / 'public-domain-v1'
if not (PD_DIR / 'manifest.json').is_file():
    subprocess.run([sys.executable, '-m', 'bloom.dataset', 'build', 'datasets/public-domain-v1/sources.json', str(PD_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'bloom.dataset', 'verify', str(PD_DIR)], check=True)

def read_jsonl_texts(path):
    texts = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                texts.append(json.loads(line)['text'])
    return texts

train_texts = read_jsonl_texts(PD_DIR / 'data' / 'train-00000.jsonl')
val_texts = read_jsonl_texts(PD_DIR / 'data' / 'validation-00000.jsonl')

if USE_TINYSTORIES:
    from datasets import load_dataset
    ds_train = load_dataset('roneneldan/TinyStories', split=f'train[:{TINYSTORIES_MAX_STORIES}]')
    ds_val = load_dataset('roneneldan/TinyStories', split=f'validation[:{TINYSTORIES_VAL_STORIES}]')
    train_texts.extend(row['text'] for row in ds_train if row['text'].strip())
    val_texts.extend(row['text'] for row in ds_val if row['text'].strip())

print(f'Train docs: {len(train_texts)} | Val docs: {len(val_texts)}')

from huggingface_hub import HfApi, snapshot_download
import shutil

PARENT_REVISION = HfApi().model_info(HF_REPO_ID).sha
if not PARENT_REVISION:
    raise RuntimeError('Could not resolve parent revision')
PARENT_DIR = Path(snapshot_download(repo_id=HF_REPO_ID, revision=PARENT_REVISION,
    allow_patterns=['config.json','tokenizer.json','model.safetensors','metrics.json']))
parent_metrics = json.loads((PARENT_DIR / 'metrics.json').read_text())
cfg_raw = json.loads((PARENT_DIR / 'config.json').read_text())
VOCAB_SIZE = cfg_raw['vocab_size']
CONTEXT = cfg_raw['context']
tok_path = OUT_DIR / 'tokenizer.json'
shutil.copyfile(PARENT_DIR / 'tokenizer.json', tok_path)
tokenizer = Tokenizer.from_file(str(tok_path))
if tokenizer.get_vocab_size() != VOCAB_SIZE:
    raise ValueError('Parent tokenizer/config mismatch')
eot_id = tokenizer.token_to_id('<|endoftext|>')
if eot_id is None:
    raise ValueError('Missing original end-of-text token')
print('Parent revision:', PARENT_REVISION, '| original vocab:', VOCAB_SIZE)
print('Reported parent training step:', parent_metrics.get('step'))

In [ ]:
# 4. Pack windows and load validated parent weights
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
from bloom.model import Bloom, Config
from bloom.checkpoint import config_hash, save_weights, load_weights

def encode_windows(texts, tok, context, eot):
    buf = []
    for t in texts:
        buf.extend(tok.encode(t).ids)
        buf.append(eot)
    arr = np.array(buf, dtype=np.int64)
    n_win = (len(arr) - 1) // context
    arr = arr[:n_win * context + 1]
    windows = np.lib.stride_tricks.sliding_window_view(arr, context + 1)[::context]
    x = torch.from_numpy(windows[:, :-1].copy())
    y = torch.from_numpy(windows[:, 1:].copy())
    return TensorDataset(x, y)

train_ds = encode_windows(train_texts, tokenizer, CONTEXT, eot_id)
val_ds = encode_windows(val_texts, tokenizer, CONTEXT, eot_id)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
print(f'Train windows: {len(train_ds)} (~{len(train_ds)*CONTEXT:,} tokens) | Val windows: {len(val_ds)}')

config = Config(
    vocab_size=tokenizer.get_vocab_size(),
    context=CONTEXT,
    width=cfg_raw['width'],
    heads=cfg_raw['heads'],
    layers=cfg_raw['layers'],
)
torch.manual_seed(SEED)
model = Bloom(config)
if parent_metrics.get('config_sha256') != config_hash(config):
    raise ValueError('Parent configuration hash mismatch')
parent_sha = parent_metrics['weights_sha256']
model.load_state_dict(load_weights(PARENT_DIR / 'model.safetensors', parent_sha, model.state_dict()), strict=True)
model.to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f'Restored Bloom: {n_params:,} parameters | config_sha256: {config_hash(config)}')

In [ ]:
# 5. Train with periodic SafeTensors checkpoints saved to Google Drive
import math, time
from itertools import islice
from torch.nn import functional as F

def evaluate(m, loader, device, max_batches):
    m.eval()
    total, count = 0.0, 0
    with torch.no_grad():
        for x, y in islice(loader, max_batches):
            x, y = x.to(device), y.to(device)
            with torch.amp.autocast('cuda', dtype=torch.float16, enabled=(device == 'cuda')):
                logits = m(x)
                loss = F.cross_entropy(logits.flatten(0, 1), y.flatten(), reduction='sum')
            total += loss.item()
            count += y.numel()
    return total / count, count

def save_ckpt(step, val_loss, train_loss, elapsed, tokens_seen):
    cpu_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
    ckpt_file = OUT_DIR / f'step-{step:06d}.safetensors'
    sha = save_weights(ckpt_file, cpu_state, cpu_state)
    latest_file = OUT_DIR / 'latest.safetensors'
    save_weights(latest_file, cpu_state, cpu_state)
    (OUT_DIR / 'config.json').write_text(json.dumps(config.__dict__, indent=2, sort_keys=True) + '\n')
    meta = {
        'step': int(parent_metrics.get('step', 0)) + step,
        'additional_steps': step,
        'parent_revision': PARENT_REVISION,
        'parent_weights_sha256': parent_sha,
        'parent_reported_train_tokens': parent_metrics.get('train_tokens'),
        'parent_val_loss_same_windows': round(val_before, 4),
        'optimizer_state_restored': False,
        'parameters': n_params,
        'config_sha256': config_hash(config),
        'weights_sha256': sha,
        'train_tokens': tokens_seen,
        'train_tokens_scope': 'this continuation only',
        'last_train_loss': round(train_loss, 4),
        'val_loss': round(val_loss, 4),
        'elapsed_seconds': round(elapsed, 2),
        'use_tinystories': USE_TINYSTORIES,
        'device': DEVICE,
    }
    (OUT_DIR / 'metrics.json').write_text(json.dumps(meta, indent=2, sort_keys=True) + '\n')
    # Private run state; never load untrusted optimizer pickle files.
    torch.save({'optimizer': optimizer.state_dict(), 'scaler': scaler.state_dict(),
                'step': step, 'tokens_seen': tokens_seen, 'parent_revision': PARENT_REVISION},
               OUT_DIR / 'local-optimizer-state.pt')
    return sha

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.1, betas=(0.9, 0.95))
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))

def lr_for_step(step):
    if step < WARMUP_STEPS:
        return LEARNING_RATE * (step + 1) / max(1, WARMUP_STEPS)
    progress = (step - WARMUP_STEPS) / max(1, STEPS - WARMUP_STEPS)
    return LEARNING_RATE * (0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progress)))

val_before, val_tokens = evaluate(model, val_loader, DEVICE, EVAL_BATCHES)
print(f'Initial val loss: {val_before:.4f} ({val_tokens} tokens)')

it = iter(train_loader)
tokens_seen = 0
t0 = time.perf_counter()
last_loss = val_before

for step in range(1, STEPS + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    for pg in optimizer.param_groups:
        pg['lr'] = lr_for_step(step - 1)
    step_loss = 0.0
    for _ in range(GRAD_ACCUM):
        batch = next(it, None)
        if batch is None:
            it = iter(train_loader)
            batch = next(it)
        x, y = batch[0].to(DEVICE), batch[1].to(DEVICE)
        with torch.amp.autocast('cuda', dtype=torch.float16, enabled=(DEVICE == 'cuda')):
            loss = F.cross_entropy(model(x).flatten(0, 1), y.flatten()) / GRAD_ACCUM
        scaler.scale(loss).backward()
        step_loss += loss.item()
        tokens_seen += y.numel()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()
    last_loss = step_loss

    if step == 1 or step % 100 == 0 or step == STEPS:
        dt = time.perf_counter() - t0
        print(f'step {step:4d}/{STEPS} | loss {last_loss:.4f} | {tokens_seen/max(dt,1e-6):,.0f} tok/s | {dt:.1f}s')
    if step % SAVE_EVERY == 0 or step == STEPS:
        vloss, _ = evaluate(model, val_loader, DEVICE, EVAL_BATCHES)
        sha = save_ckpt(step, vloss, last_loss, time.perf_counter() - t0, tokens_seen)
        print(f'  -> Saved checkpoint step {step} | val_loss {vloss:.4f} | sha256 {sha[:12]}...')

val_after, _ = evaluate(model, val_loader, DEVICE, EVAL_BATCHES)
print(f'Done in {time.perf_counter() - t0:.1f}s | Val loss: {val_before:.4f} -> {val_after:.4f}')

In [ ]:
# 6. Generate sample completions
def generate(prompt, max_new_tokens=128, temperature=0.8, top_k=40):
    model.eval()
    ids = tokenizer.encode(prompt).ids or [eot_id]
    idx = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    with torch.no_grad():
        for _ in range(max_new_tokens):
            ctx = idx[:, -config.context:]
            logits = model(ctx)[:, -1, :] / max(temperature, 1e-5)
            if top_k:
                vals, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < vals[:, [-1]]] = float('-inf')
            probs = torch.softmax(logits, dim=-1)
            nxt = torch.multinomial(probs, num_samples=1)
            if nxt.item() == eot_id:
                break
            idx = torch.cat([idx, nxt], dim=1)
    return tokenizer.decode(idx[0].tolist())

for p in ['Once upon a time', 'Alice was', 'The little']:
    print('=== PROMPT:', repr(p), '===')
    print(generate(p))
    print()

In [ ]:
# 7. Review candidate — no upload
metrics = json.loads((OUT_DIR / 'metrics.json').read_text())
print(json.dumps(metrics, indent=2))
print('Candidate directory:', OUT_DIR)
print('Compare parent and candidate validation losses on the same windows and inspect samples.')
print('Publish compatible config, original tokenizer, weights and metrics together only after review.')
